In [ ]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
sys.path.append(base_dir)
from Models.BERT_Model.BERT_Model import BERT_Lag
from Models.BERT_Model.train_BERT import train_loop , estimate_loss
from Models.Configs import BERTConfig, TrainConfig
from Datasets.DataLoader import CombinedBinDataLoader

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
print(F"Device set to {device}")


In [ ]:
model_config = BERTConfig()
train_config = TrainConfig()
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = BERT_Lag(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')


In [ ]:
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

In [ ]:
torch.cuda.empty_cache()
history = train_loop(model,
                     optimizer,
                     scheduler,
                     scaler,
                     device,
                     train_loader,
                     val_loader,
                     train_config,
                     model_config)

In [ ]:
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_BERT_combined_loss.pt"
torch.save({
            "step": 3623,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict()
        }, path)

In [ ]:
loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, 100)
print(f"Val FWD: {loss_fwd:.4f} | PPL FWD: {ppl_fwd:.2f} | PPL BWD: {ppl_bwd:.2f} | Val BWD: {loss_bwd:.4f}")

In [ ]:
import tiktoken

def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=50, device=device):
    enc = tiktoken.get_encoding("gpt2")
    tokens = enc.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)  # [1, T]

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -model.config.block_size:]

            B, N = x_cond.size()
            pos = torch.arange(0, N, dtype=torch.long, device=device)

            with torch.amp.autocast('cuda'):
                h = model.transformer.drop(
                    model.transformer.wte(x_cond) + model.transformer.wpe(pos)
                )
                for block in model.transformer.h:
                    h = block(h, mask=True)
                h = model.transformer.ln_f(h)
                logits = model.lm_head(h)  # [1, T, vocab_size]

            logits = logits[:, -1, :].float() / temperature  # [1, vocab_size]

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)

    model.train()
    return enc.decode(x[0].tolist())

print(generate(model, """One, Two, Three, Four, Five""", max_new_tokens=100, temperature=1, top_k=50))